# Creador de Mundos RPG — Pack de Arte Pintado (Opción 1)

Genera en GPU T4 con **`Lykon/dreamshaper-xl-v2-turbo`** (modelo open-source gratuito, estilo pintura fantástica tipo *Lineage 2*):
- **1 portada épica** (`1024×576` WebP)
- **15 retratos** de sabios, NPCs y compañeros (`256×256` WebP)
- **23 iconos RPG** de habilidades y pociones (`128×128` WebP)

**Ejecución en 4 celdas (~2-3 minutos total en T4).**
La Celda 4 empaqueta `arte_rpg_valdris.zip` con `manifest.json` listo para descomprimir en `assets/arte/` de tu repo.

In [ ]:
# 1. Verificar GPU y montar Google Drive
import os
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
from google.colab import drive
drive.mount('/content/drive')

WORK_DIR = '/content/drive/MyDrive/Creador-de-mundos-Rpg-Colab'
PROMPTS_PATH = f'{WORK_DIR}/assets/arte/prompts_arte.json'
SCRIPT_PATH = f'{WORK_DIR}/tools/generar-arte-ia.py'
OUT_DIR = '/content/arte_out'
os.makedirs(OUT_DIR, exist_ok=True)
assert os.path.exists(PROMPTS_PATH), f'Falta {PROMPTS_PATH}'
assert os.path.exists(SCRIPT_PATH), f'Falta {SCRIPT_PATH}'
print('Drive listo:', WORK_DIR)

In [ ]:
# 2. Instalar dependencias (DreamShaper XL Turbo via diffusers)
!pip -q install -U diffusers transformers accelerate safetensors pillow

In [ ]:
# 3. Generar las 39 ilustraciones (1 portada + 15 retratos + 23 iconos)
!python "{SCRIPT_PATH}" --prompts "{PROMPTS_PATH}" --out "{OUT_DIR}" --steps 6 --guidance 2.0

In [ ]:
# 4. Copiar a Drive, mostrar muestra visual y descargar ZIP para assets/arte/
import glob, json, shutil
from IPython.display import Image as ColabImage, display
from google.colab import files

# Sincronizar resultados en Drive como respaldo
DRIVE_OUT = f'{WORK_DIR}/assets/arte'
shutil.copytree(OUT_DIR, DRIVE_OUT, dirs_exist_ok=True)

with open(f'{OUT_DIR}/manifest.json', 'r', encoding='utf-8') as f:
    m = json.load(f)
print('Manifest:', {'portada': bool(m['portada']), 'retratos': len(m['retratos']), 'iconos': len(m['iconos'])})

# Vista previa rápida en pantalla (IPython.display.Image evita errores por actualización en caliente de Pillow)
muestras = [f'{OUT_DIR}/portada/portada_valdris.webp'] + sorted(glob.glob(f'{OUT_DIR}/retratos/*.webp'))[:4] + sorted(glob.glob(f'{OUT_DIR}/iconos/*.webp'))[:6]
for p in muestras:
    if os.path.exists(p):
        print(' ->', os.path.relpath(p, OUT_DIR))
        display(ColabImage(filename=p, width=320))

zip_path = shutil.make_archive('/content/arte_rpg_valdris', 'zip', OUT_DIR)
print('ZIP listo:', zip_path)
files.download(zip_path)